# 🔋 GoodWe ChargeOps Assistant
## Chatbot com IA para Gerenciamento de Eletropostos

---

**Disciplina:** Prompt and Artificial Intelligence
**Curso:** Análise e Desenvolvimento de Sistemas — FIAP  
**Semestre:** 1º Semestre — 2026  
**Desafio:** EV Challenge 2026 — GoodWe

---

## 👥 Integrantes

| Nome | RM |
|------|-----|
| *Ana Beatriz Berbel Marini* | RM574176 |
| *Gustavo Bonamico Piccoli* | RM569984 |
| *Julian Nayde Moncoski* | RM572603 |
| *Marcelo Francisco Josafá Ribeiro Martins* | RM573905 |
| *Maria Eduarda Medeiros Lemos* | RM574094 |
| *Pietro Lorande da Silva* | RM569125 |


# 🔌 GoodWe Assist — Chatbot EV Challenge 2026
### Sprint 2 — Desenvolvimento e Entrega

---

| Item | Detalhe |
|---|---|
| **Projeto** | EV Challenge 2026 — GoodWe / FIAP |
| **Persona** | Gestor de Eletroposto Comercial |
| **Escopo** | ChargeGrid Intelligence |
| **Técnicas** | RAG · Few-Shot Prompting · Histórico de Conversa |
| **Modelo** | Qwen/Qwen2.5-7B-Instruct (Hugging Face) |
| **Interface** | Gradio ChatInterface |
| **Ambiente** | Kaggle Notebook |

---

### Arquitetura do Sistema

```
Usuário
   │  pergunta
   ▼
ChromaDB ──► recupera 3 trechos relevantes da base GoodWe
   │  contexto
   ▼
System Prompt (few-shot) + Contexto RAG + Histórico
   │  mensagens
   ▼
Qwen2.5-7B-Instruct (Hugging Face Inference API)
   │  resposta
   ▼
Gradio ChatInterface
```


## ⚙️ Célula 1 — Instalação das Dependências

In [1]:
!pip install chromadb pypdf gradio huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.0 MB/s eta 0:00:00
ERROR: pip's dependency r

## 🔑 Célula 2 — Autenticação

> **Kaggle:** Clique em **Add-ons → Secrets**, adicione `HUGGING_FACE_API_KEY` com seu token HF.  
> **Colab:** Clique no ícone 🔑 (Secrets) no menu lateral e adicione a mesma variável.  
> Obtenha seu token gratuito em: https://huggingface.co/settings/tokens


In [2]:
import os
from huggingface_hub import InferenceClient

# ── Carrega o token de forma compatível com Kaggle e Colab ────────────────────
def _get_token() -> str:
    # 1. Variável de ambiente (Kaggle Secrets injeta automaticamente)
    token = os.environ.get("HUGGING_FACE_API_KEY", "")
    if token:
        return token

    # 2. Kaggle Secrets via SDK
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HUGGING_FACE_API_KEY")
    except Exception:
        pass

    # 3. Google Colab Secrets via userdata
    try:
        from google.colab import userdata
        return userdata.get("HUGGING_FACE_API_KEY")
    except Exception:
        pass

    raise EnvironmentError(
        "Token não encontrado. Adicione HUGGING_FACE_API_KEY nos Secrets "
        "do Kaggle ou do Colab antes de continuar."
    )

token_hf = _get_token()

MODELO = "Qwen/Qwen2.5-7B-Instruct"
client  = InferenceClient(model=MODELO, token=token_hf)

print(f"✅ Autenticado com sucesso! Modelo: {MODELO}")


✅ Autenticado com sucesso! Modelo: Qwen/Qwen2.5-7B-Instruct


## 📚 Célula 3 — Base de Conhecimento GoodWe

Carrega os três PDFs técnicos, extrai o texto e cria chunks de 1.000 caracteres para indexação no ChromaDB.

> **Kaggle:** suba os PDFs como dataset e ajuste `PASTA_PDFS` abaixo.  
> **Colab:** carregue os arquivos via upload (ícone de pasta → upload) e defina `PASTA_PDFS = "/content"`.


In [12]:
from pypdf import PdfReader
import textwrap
import os

# ── Configure o caminho onde os PDFs estão ───────────────────────────────────
# Kaggle: o path correto é /kaggle/input/<nome-do-dataset>/
# O nome do dataset aparece no painel Input do notebook (ex: "Dataset_Goodwe")
PASTA_PDFS = "/kaggle/input/datasets/bpgustavo/dataset-goodwe"
# PASTA_PDFS = "/content"  # descomente se estiver no Colab

# ── Nome dos arquivos exatamente como aparecem no dataset ────────────────────
# ATENÇÃO: use o nome exato — incluindo espaços se houver.
# Verifique no painel Input → Dataset_Goodwe os nomes listados.
ARQUIVOS_PDF = {
    "Datasheet PT": "GW_HCA-G2_Datasheet-PT.pdf",
    "Manual PT":    "GW_HCA-G2_User-Manual-PT.pdf",
    "Mapa MODBUS":  "Mapa MODBUS_HCA G2.pdf",   # nome com espaços, igual ao Kaggle
}

# ── Valida os caminhos antes de tentar ler ───────────────────────────────────
print(f"📁 Pasta dos PDFs: {PASTA_PDFS}")
if not os.path.isdir(PASTA_PDFS):
    raise FileNotFoundError(
        f"Pasta não encontrada: '{PASTA_PDFS}'\n"
        "Verifique o nome exato do dataset no painel Input do Kaggle e ajuste PASTA_PDFS."
    )

for nome, arquivo in ARQUIVOS_PDF.items():
    caminho = os.path.join(PASTA_PDFS, arquivo)
    if not os.path.isfile(caminho):
        # lista os arquivos disponíveis para facilitar o diagnóstico
        disponiveis = os.listdir(PASTA_PDFS)
        raise FileNotFoundError(
            f"Arquivo não encontrado: '{caminho}'\n"
            f"Arquivos disponíveis em '{PASTA_PDFS}':\n  " + "\n  ".join(disponiveis)
        )

print("✅ Todos os arquivos encontrados!\n")


def extrair_texto(caminho: str) -> str:
    leitor = PdfReader(caminho)
    return "\n".join(p.extract_text() or "" for p in leitor.pages)

def chunk_texto(texto: str, largura: int = 1000) -> list[str]:
    return textwrap.wrap(texto, width=largura)

textos_completos: list[str] = []
todos_chunks:     list[str] = []
todos_ids:        list[str] = []

chunk_global = 0
for nome, arquivo in ARQUIVOS_PDF.items():
    caminho = os.path.join(PASTA_PDFS, arquivo)
    print(f"📄 Lendo: {nome} ({arquivo})...")

    texto = extrair_texto(caminho)
    textos_completos.append(texto)

    chunks = chunk_texto(texto)
    ids    = [f"chunk_{chunk_global + i:04d}" for i in range(len(chunks))]

    todos_chunks.extend(chunks)
    todos_ids.extend(ids)
    chunk_global += len(chunks)

    print(f"   ✅ {len(chunks)} blocos gerados.")

print(f"\n✅ Total: {len(todos_chunks)} chunks de {len(ARQUIVOS_PDF)} documentos.")

# BASE_GOODWE é usada na indexação e no RAG
BASE_GOODWE   = textos_completos
CHUNKS_GOODWE = todos_chunks
IDS_CHUNKS    = todos_ids


📁 Pasta dos PDFs: /kaggle/input/datasets/bpgustavo/dataset-goodwe
✅ Todos os arquivos encontrados!

📄 Lendo: Datasheet PT (GW_HCA-G2_Datasheet-PT.pdf)...
   ✅ 4 blocos gerados.
📄 Lendo: Manual PT (GW_HCA-G2_User-Manual-PT.pdf)...
   ✅ 71 blocos gerados.
📄 Lendo: Mapa MODBUS (Mapa MODBUS_HCA G2.pdf)...
   ✅ 23 blocos gerados.

✅ Total: 98 chunks de 3 documentos.


## 🗄️ Célula 4 — Indexação no ChromaDB (Banco Vetorial)

In [13]:
import chromadb

cliente_db = chromadb.Client()

# Garante coleção limpa a cada execução
try:
    cliente_db.delete_collection("base_goodwe")
except Exception:
    pass

colecao = cliente_db.create_collection(name="base_goodwe")

print(f"Vetorizando e indexando {len(CHUNKS_GOODWE)} chunks...")

# Indexa em lotes de 100 para evitar timeout no ChromaDB
LOTE = 100
for inicio in range(0, len(CHUNKS_GOODWE), LOTE):
    fim = inicio + LOTE
    colecao.add(
        documents=CHUNKS_GOODWE[inicio:fim],
        ids=IDS_CHUNKS[inicio:fim],
    )

print(f"✅ ChromaDB pronto! {colecao.count()} chunks indexados.")


Vetorizando e indexando 98 chunks...
✅ ChromaDB pronto! 98 chunks indexados.


## 💬 Célula 5 — System Prompt com Few-Shot Prompting

**Técnicas aplicadas:**
- **System Prompt** focado na persona Gestor de Eletroposto Comercial
- **Few-Shot Prompting**: exemplos de pergunta/resposta calibram o tom e o formato esperados


In [14]:
SYSTEM_PROMPT = """Você é o GoodWe Assist, assistente especializado em operação de eletropostos comerciais GoodWe. Responda em português, de forma direta e prática, baseando-se apenas no contexto fornecido. Se não souber, indique o suporte GoodWe.

Exemplos do comportamento esperado:

Pergunta: "Como evitar multas por ultrapassagem de demanda?"
Resposta: "O ChargeGrid Intelligence faz load balancing automático a cada 30 segundos: quando a demanda se aproxima do limite contratado, o sistema reduz proporcionalmente a potência de cada carregador ativo. Nenhuma ação manual é necessária."

Pergunta: "Qual modelo usar para um posto de alto fluxo?"
Resposta: "Para alto fluxo, o EV-DC60 é o indicado: 60 kW DC, carrega até 80% em ~50 minutos, aceita Pix e cartão diretamente no totem. Ideal para postos de combustível e corredores de viagem."

Agora responda a pergunta do usuário com base no contexto abaixo:

[CONTEXTO]
{contexto}
[/CONTEXTO]"""

print("✅ System Prompt com few-shot configurado!")


✅ System Prompt com few-shot configurado!


## 🤖 Célula 6 — Função do Chatbot (RAG + Histórico de Conversa)

In [15]:
import time

MAX_TENTATIVAS  = 3
PAUSA_RETRY_S   = 8   # segundos entre tentativas


def chatbot_goodwe(pergunta: str, historico: list) -> str:
    """
    Chatbot GoodWe com RAG + memória de conversa + retry automático.

    Fluxo:
      1. RETRIEVAL    : ChromaDB recupera os 3 trechos mais relevantes
      2. AUGMENTATION : monta o system prompt com o contexto recuperado
      3. HISTÓRICO    : inclui turnos anteriores para manter coerência
      4. GENERATION   : Qwen2.5 gera a resposta (com retry em caso de erro)
    """

    # ── 1. RETRIEVAL ──────────────────────────────────────────────────────────
    busca   = colecao.query(query_texts=[pergunta], n_results=3)
    contexto = "\n\n".join(busca["documents"][0])

    # ── 2. AUGMENTATION ───────────────────────────────────────────────────────
    system_final = SYSTEM_PROMPT.format(contexto=contexto)
    mensagens    = [{"role": "system", "content": system_final}]

    # ── 3. HISTÓRICO DE CONVERSA ──────────────────────────────────────────────
    for turno in historico:
        if isinstance(turno, (list, tuple)) and len(turno) == 2:
            if turno[0]:
                mensagens.append({"role": "user",      "content": turno[0]})
            if turno[1]:
                mensagens.append({"role": "assistant", "content": turno[1]})

    mensagens.append({"role": "user", "content": pergunta})

    # ── 4. GENERATION com retry ───────────────────────────────────────────────
    ultimo_erro = None
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            resposta_obj = client.chat_completion(
                messages=mensagens,
                max_tokens=600,
                temperature=0.2,
            )
            return resposta_obj.choices[0].message.content

        except Exception as e:
            ultimo_erro = e
            codigo = getattr(getattr(e, "response", None), "status_code", "?")
            print(f"  ⚠️  Tentativa {tentativa}/{MAX_TENTATIVAS} falhou "
                  f"(HTTP {codigo}): {type(e).__name__}")

            if tentativa < MAX_TENTATIVAS:
                print(f"     Aguardando {PAUSA_RETRY_S}s antes de tentar novamente...")
                time.sleep(PAUSA_RETRY_S)

    return (
        f"❌ Não foi possível obter resposta após {MAX_TENTATIVAS} tentativas. "
        f"Erro: {ultimo_erro}. "
        "Verifique sua conexão, o token HF e tente novamente em alguns instantes."
    )


print("✅ Função do chatbot pronta! (retry automático ativo)")


✅ Função do chatbot pronta! (retry automático ativo)


## 🧪 Célula 7 — Execução dos 5 Casos de Teste (Sprint 1)

In [16]:
import time

CASOS_DE_TESTE = [
    {
        "id": 1,
        "persona": "Operador Comercial",
        "pergunta": "Como o ChargeGrid Intelligence distribui a potência entre os carregadores?",
        "resposta_esperada": "Load balancing dinâmico: reduz proporcionalmente a potência de cada carregador quando a demanda supera o limite contratado.",
    },
    {
        "id": 2,
        "persona": "Operador Comercial",
        "pergunta": "Quais formas de pagamento são suportadas nos eletropostos?",
        "resposta_esperada": "Pix, cartão de crédito/débito e carteiras digitais. Faturamento configurável por kWh, tempo, taxa fixa ou assinatura.",
    },
    {
        "id": 3,
        "persona": "Técnico",
        "pergunta": "Quais são os requisitos elétricos para instalar o modelo EV-AC22?",
        "resposta_esperada": "Alimentação trifásica 380V, cabo mínimo 6mm², quadro exclusivo. Instalação por eletricista NR-10 com ART/RRT.",
    },
    {
        "id": 4,
        "persona": "Técnico",
        "pergunta": "O que significa o Erro E03 e como resolver?",
        "resposta_esperada": "Temperatura elevada. Garantir 20 cm livres ao redor do equipamento. Se persistir, acionar suporte GoodWe.",
    },
]

PAUSA_ENTRE_TESTES_S = 5   # evita rate-limit da Inference API gratuita

print("🧪 Executando casos de teste...\n")
print("=" * 70)

resultados = []

for caso in CASOS_DE_TESTE:
    print(f"\n[Teste {caso['id']}] Persona: {caso['persona']}")
    print(f"❓ Pergunta : {caso['pergunta']}")
    print(f"✅ Esperada : {caso['resposta_esperada']}")

    resposta_obtida = chatbot_goodwe(caso["pergunta"], [])

    print(f"🤖 Obtida   : {resposta_obtida[:400]}")
    print("-" * 70)

    resultados.append({**caso, "resposta_obtida": resposta_obtida})

    # Pausa entre chamadas para não disparar o rate-limit
    if caso["id"] < len(CASOS_DE_TESTE):
        print(f"   ⏳ Aguardando {PAUSA_ENTRE_TESTES_S}s antes do próximo teste...")
        time.sleep(PAUSA_ENTRE_TESTES_S)

print("\n✅ Testes concluídos!")


🧪 Executando casos de teste...


[Teste 1] Persona: Operador Comercial
❓ Pergunta : Como o ChargeGrid Intelligence distribui a potência entre os carregadores?
✅ Esperada : Load balancing dinâmico: reduz proporcionalmente a potência de cada carregador quando a demanda supera o limite contratado.
🤖 Obtida   : O ChargeGrid Intelligence distribui a potência entre os carregadores de forma automática e balanceada. A cada 30 segundos, o sistema verifica a demanda total e ajusta a potência de cada carregador para evitar ultrapassar o limite contratado. Isso é feito de forma proporcional, garantindo que nenhum carregador seja sobrecarregado e que a distribuição seja justa entre todos. Nenhuma ação manual é ne
----------------------------------------------------------------------
   ⏳ Aguardando 5s antes do próximo teste...

[Teste 2] Persona: Operador Comercial
❓ Pergunta : Quais formas de pagamento são suportadas nos eletropostos?
✅ Esperada : Pix, cartão de crédito/débito e carteiras digitais

## 📋 Célula 8 — Tabela de Resultados dos Testes

Após executar os testes, preenchemos a coluna **Avaliação** comparando a resposta esperada com a obtida.

| # | Persona | Pergunta (resumo) | Avaliação |
|---|---|---|---|
| 1 | Operador Comercial | Distribuição de potência entre carregadores | ✅ |
| 2 | Operador Comercial | Formas de pagamento suportadas | ✅ |
| 3 | Técnico | Requisitos elétricos do EV-AC22 | ✅/⚠️ |
| 4 | Técnico | Erro E03 — significado e solução | ✅ |

> **Legenda:** ✅ Adequada · ⚠️ Parcialmente adequada · ❌ Inadequada
(OBS.: Na pergunta 3, avaliei com ✅/⚠️, pois de primeira/no teste, o chatbot sempre está respondendo algo um pouco estranho ou com alguns caracteres em mandarim. Porém, a partir da segunda resposta no Gradio, ele sempre responde muito bem com um texto bem mais rico e completo.)

## 🖥️ Célula 9 — Interface Gradio

In [17]:
import gradio as gr

EXEMPLOS = [
    "Como o ChargeGrid Intelligence distribui a potência entre os carregadores?",
    "Quais formas de pagamento são aceitas nos eletropostos GoodWe?",
    "Qual modelo de carregador indicar para um posto de alto fluxo?",
    "O que significa o Erro E03 e como resolver?",
    "Quais são os requisitos elétricos para instalar o EV-AC22?",
    "Como integrar o sistema GoodWe com meu software financeiro?",
]

interface = gr.ChatInterface(
    fn=chatbot_goodwe,
    title="🔌 GoodWe Assist — EV Challenge 2026",
    description=(
        "**Assistente para Gestores de Eletropostos Comerciais GoodWe**\n\n"
        "Pergunte sobre: orquestração de potência · faturamento · modelos de equipamento · "
        "instalação elétrica · erros e manutenção · integração via API"
    ),
    theme=gr.themes.Soft(primary_hue="green", secondary_hue="emerald"),
    examples=EXEMPLOS,
    cache_examples=False,
)

interface.launch(share=True, debug=False)


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://8d3b22894a9d6a8e2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 📄 Instruções de Execução

### Pré-requisitos
- Conta Kaggle **ou** Google Colab
- Token da Hugging Face gratuito: https://huggingface.co/settings/tokens

### Como executar no Kaggle
1. Abra `goodwe-sprint2.ipynb` no Kaggle
2. Clique em **Add-ons → Secrets** e adicione:
   - Nome: `HUGGING_FACE_API_KEY` | Valor: seu token (`hf_...`)
3. Suba os três PDFs como dataset e ajuste `PASTA_PDFS` na Célula 3
4. Execute todas as células: **Run All**

### Como executar no Google Colab
1. Abra o notebook no Colab
2. Clique no ícone 🔑 (Secrets) e adicione `HUGGING_FACE_API_KEY`
3. Faça upload dos PDFs e defina `PASTA_PDFS = "/content"` na Célula 3
4. Execute todas as células: **Runtime → Run all**

### Dependências
```
chromadb · gradio · huggingface_hub · pypdf
```
Instaladas automaticamente pela Célula 1.

### Variáveis de Ambiente
| Variável | Onde configurar | Descrição |
|---|---|---|
| `HUGGING_FACE_API_KEY` | Kaggle Secrets / Colab Secrets | Token de acesso à Hugging Face Inference API |

> ⚠️ **Nunca exponha sua API Key no código ou em repositório público.**

### Melhorias aplicadas nesta versão
| # | Problema | Correção |
|---|---|---|
| 1 | `kaggle_secrets` quebrava no Colab | Detecção automática do ambiente (Kaggle, Colab ou env var) |
| 2 | `HTTPStatusError` sem tratamento | Retry automático (3 tentativas, 8s de pausa entre elas) |
| 3 | Rate-limit entre os 5 testes | Pausa de 5s entre cada chamada de teste |
| 4 | ChromaDB indexava textos inteiros | Indexação por chunks (melhor precisão no retrieval) |
| 5 | Coleção duplicada ao re-executar | `delete_collection` antes de criar garante estado limpo |
| 6 | Nome do PDF com espaço | Nomes dos arquivos mantidos exatamente como estão no Kaggle |
| 7 | Path errado (`datasets/bpgustavo/...`) | Corrigido para `/kaggle/input/Dataset_Goodwe` + validação prévia com mensagem de erro clara |

### Técnicas implementadas
- ✅ **RAG** com ChromaDB — respostas baseadas nos documentos técnicos GoodWe
- ✅ **Few-Shot Prompting** — exemplos no system prompt calibram tom e formato
- ✅ **Histórico de conversa** — memória de múltiplos turnos
- ✅ **API Key via Secrets** — sem exposição de credenciais no código
- ✅ **Retry com back-off** — resiliência a erros transientes da Inference API
